In [1]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [2]:
import sys, xgboost as xgb
print(sys.executable)
print(xgb.__version__)
print(xgb.__file__)

c:\Users\barna\OneDrive\Desktop\HOUSING MLE\venv\Scripts\python.exe
3.2.0
c:\Users\barna\OneDrive\Desktop\HOUSING MLE\venv\Lib\site-packages\xgboost\__init__.py


In [3]:
! pip install optuna
! pip install mlflow


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\barna\OneDrive\Desktop\HOUSING MLE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
train_df = pd.read_csv("../data/processed/feature_eng_train.csv")
eval_df = pd.read_csv("../data/processed/feature_eng_eval.csv")

target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [6]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [7]:
# RUN OPTUNA

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

c:\Users\barna\OneDrive\Desktop\HOUSING MLE\venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[I 2026-05-21 05:52:19,125] A new study created in memory with name: no-name-d688c474-5180-4fb2-a91d-fac4e8d285fc


[I 2026-05-21 05:52:49,796] Trial 0 finished with value: 71444.89463837339 and parameters: {'n_estimators': 910, 'max_depth': 5, 'learning_rate': 0.03552171122446907, 'subsample': 0.6288544435919976, 'colsample_bytree': 0.8117405716645425, 'min_child_weight': 4, 'gamma': 4.5760311224969135, 'reg_alpha': 0.01065262882761141, 'reg_lambda': 9.909304921183198}. Best is trial 0 with value: 71444.89463837339.
[I 2026-05-21 05:53:34,001] Trial 1 finished with value: 73426.37824418463 and parameters: {'n_estimators': 863, 'max_depth': 6, 'learning_rate': 0.016611845818527444, 'subsample': 0.5272539760236633, 'colsample_bytree': 0.9057517792783811, 'min_child_weight': 2, 'gamma': 4.215080580176121, 'reg_alpha': 0.006321173599411017, 'reg_lambda': 1.4615619628356436e-08}. Best is trial 0 with value: 71444.89463837339.
[I 2026-05-21 05:53:50,049] Trial 2 finished with value: 75254.85006175863 and parameters: {'n_estimators': 227, 'max_depth': 8, 'learning_rate': 0.15099394874888833, 'subsample': 

Best params: {'n_estimators': 914, 'max_depth': 7, 'learning_rate': 0.02532588252282468, 'subsample': 0.5159409860148595, 'colsample_bytree': 0.5969539233532142, 'min_child_weight': 6, 'gamma': 4.122053580520875, 'reg_alpha': 0.0006638604127844734, 'reg_lambda': 0.1295504475123555}


In [9]:
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
r2 = float(r2_score(y_eval, y_pred))

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 31600.58731161515
RMSE: 70039.5177269631
R²: 0.9620906511215904
